In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:22px;}

</style>
"""))

# 1.패키지

In [16]:
# pyrhon-dotenv,langchain,langchain-openai,langchain-pinecon,langchain-community,docx2txt
# langchain-upstage, langchain-ollama

%pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [7]:
%pip show pandas

Name: pandas
Version: 2.3.3
Summary: Powerful data structures for data analysis, time series, and statistics
Home-page: https://pandas.pydata.org
Author: 
Author-email: The Pandas Development Team <pandas-dev@python.org>
License: BSD 3-Clause License
         
         Copyright (c) 2008-2011, AQR Capital Management, LLC, Lambda Foundry, Inc. and PyData Development Team
         All rights reserved.
         
         Copyright (c) 2011-2023, Open source contributors.
         
         Redistribution and use in source and binary forms, with or without
         modification, are permitted provided that the following conditions are met:
         
         * Redistributions of source code must retain the above copyright notice, this
           list of conditions and the following disclaimer.
         
         * Redistributions in binary form must reproduce the above copyright notice,
           this list of conditions and the following disclaimer in the documentation
           and/or o

# 2. 환경변수 설정

In [17]:
import os
from dotenv import load_dotenv
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")
OPENAI_LLM_MODEL = "gpt-4o-mini"
UPSTAGE_LLM_MODEL = "solar-pro4"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-large"
UPSTAGE_EMBEDDING_MODEL = "embedding-query"

PINECONE_INDEX_NAME = "better-rag-index"
PINECONE_INDEX_CLOUD = "aws"
PINECONE_INDEX_METRIC = 'cosine'
PINECONE_INDEX_REGION = 'us-east-1'
PINECONE_INDEX_DIMENSION =3072

# 3. 문서를 chunk로 분할하기


In [20]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader('./tex_docx/소득세법_with_markdown.docx')
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, # 1500글자 기준으로 분리
    chunk_overlap=100, 
    separators=['\n\n', '\n', ' ', '']
)
# documents = loader.load_and_split(text_splitter=text_splitter)
document = loader.load()
documents = text_splitter.split_documents(document)
print(f"총 {len(documents)}개 청크로 분리")


총 184개 청크로 분리


# 4. metadata 추가하기
- chunk 내용의 title,category, 조항

In [21]:
import re
def remove_special_chars(text:str) -> str:
    "특수문자 및 \n제거 (:, .는 그대로)"
    text = text.replace("\n", " ")
    # 한글, 영문, 숫자, 공백, . , : |-만 남기고 전부 제거
    cleaned = re.sub(r"[^가-힣a-zA-Z0-9\s\.,:|\-]", "", text).strip()
    # 불필요한 중복 공백을 제거
    cleaned = re.sub(r"\s+", " ", cleaned)
    return cleaned
content = " ▶️ **소득세 납세 의무 ** : \n\n|이 법 또는 「법인세법」에 따라 소득에 대한 소득세를 부과  "
remove_special_chars(content)

'소득세 납세 의무 : |이 법 또는 법인세법에 따라 소득에 대한 소득세를 부과'

In [22]:
remove_special_chars(documents[43].page_content)

'전문개정 2009. 12. 31. 제목개정 2014. 1. 1. 제54조의2공동사업에 대한 소득공제 등 특례 제51조의3 또는 조세특례제한법에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출납입투자출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출납입투자출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. 개정 2012. 1. 1., 2014. 1. 1. 전문개정 2009. 12. 31. 제목개정 2014. 1. 1. 제4절 세액의 계산 개정 2009. 12. 31. 제1관 세율 개정 2009. 12. 31. 제55조세율 거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액이하 종합소득산출세액이라 한다을 그 세액으로 한다. 개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31. | 종합소득 과세표준 | 세율 | |---|---| | 1,400만원 이하 | 과세표준의 6퍼센트 | | 1,400만원 초과 5,000만원 이하 | 84만원 1,400만원을 초과하는 금액의 15퍼센트 | | 5,000만원 초과 8,800만원 이하 | 624만원 5,000만원을 초과하는 금액의 24퍼센트 | | 8,800만원 초과 1억5천만원 이하 | 1,536만원 8,800만원을 초과하는 금액의 35퍼센트 | | 1억5천만원 초과 3억원 이하 | 3,706만원 1억5천만원을 초과하는 금액의 38퍼센트 | | 3억원 초과 5억원 이하 | 9,406만원 3억원을 초과하는 금액의 40퍼센트 | | 5억원 초과 10억원 이하 | 1억7,406만원 5억원을 초과하는 금액의 42퍼센트 | | 10억원 초과 | 

In [31]:
%%time
# 제목을 추출하는 함수(사용 llm은 exaone3.5:2.4b) ollama list -> ollama pull exaone3.5:2.4b
from langchain_ollama import ChatOllama
def extract_title_with_llm(content):
    "exaone 3.5를 사용하여 content의 제목을 반환"
    title_extractor_llm = ChatOllama(
        model = 'exaone3.5:2.4b',
        temperature=0.1
    )
    prompt = f"""다음 소득세법 조문의 핵심 제목을 100토큰 이내로 완벽하게 말이 되도록 추출해 주세요.
    중간에 말이 끊기면 안 되요.
    조문 : {content}"""
    ai_message = title_extractor_llm.invoke(prompt)
    title = ai_message.content.strip()
    return remove_special_chars(title)
extract_title_with_llm(remove_special_chars(documents[43].page_content))

CPU times: total: 62.5 ms
Wall time: 18.3 s


'공동사업 소득공제 특례 주된 공동사업자의 특수관계인 지출납입 등으로 인한 소득공제 한도 적용 규정.'

In [34]:
def categorize_content(content):
    "content의 카테고리를 list로 반환"
    # gpt가 meta데이터 활용한다고 하니 만들어준 카테고리
    categories = {
    '납세의무': ['납세의무', '거주자', '비거주자', '원천징수', '공동사업자', '상속인', '증여자', '신탁재산'],
    '세율계산': ['세율', '과세표준', '산출세액', '결정세액', '종합소득', '퇴직소득'],
    '근로소득': ['근로소득', '총급여', '급여', '연봉', '임금', '퇴직소득'],
    '사업소득': ['사업소득', '공동사업', '주택임대소득', '부업소득'],
    '이자배당': ['이자소득', '배당소득', '예금이자', '채권', '의제배당'],
    '양도소득': ['양도소득', '자산양도', '부동산양도', '주식양도'],
    '연금소득': ['연금소득', '공적연금', '사적연금', '연금보험'],
    '기타소득': ['기타소득', '상금', '보상금', '발명보상금', '종교인소득'],
    '공제감면': ['공제', '소득공제', '세액공제', '기본공제', '감면'],
    '비과세': ['비과세', '면제', '복무급여', '실업급여', '출산휴가급여', '장학금'],
    '신고납부': ['신고', '납부', '납세지', '신고기한', '원천징수'],
    '과세기간': ['과세기간', '과세연도', '사업연도'],
    '과세소득구분': ['종합소득', '퇴직소득', '양도소득', '금융투자소득'],
    }
    result = [] # 각 카테고리를 추가
    for category, keywords in categories.items():
        #print(category, keywords)
        # print(any([keyword in content for keyword in keywords])) : 하나라도 True인지 여부
        if any([keyword in content for keyword in keywords]):
            result.append(category)
    

In [35]:
categorize_content(documents[43].page_content)